# Setup: Shared Lamna Healthcare Lakehouse (Instructor)

Run this notebook **once per workspace** to provision the shared data that all 25 participants use in the *FabricIQ mini Hands-on* lab.

It reuses the data-loading steps from the official `setup-ontology.ipynb` (Labs 27-28), but **stops before creating any ontology** — because each participant builds their **own** ontology and data agent.

**What it creates:**
- Lakehouse `LamnaHealthcareLH` with 5 tables: `Hospitals`, `Departments`, `Rooms`, `Patients`, `VitalSignEquipment`
- Eventhouse `LamnaHealthcareEH` (KQL database) with the `VitalSignsReadings` time-series table (15 rows)

**What it does NOT create:** ontologies or data agents. Participants create those themselves.

**Idempotent** — safe to re-run. Existing infrastructure is reused; tables are overwritten; readings are skipped if already present.

> **Prerequisite:** a paid or trial Fabric capacity, and the tenant settings listed in `setup/instructor-setup-guide.md` enabled.

## Step 0: Get or create infrastructure

Checks for the shared lakehouse and eventhouse. Creates them if they don't exist.

> **Note**: The first cell installs a required package and may show dependency warnings. These warnings are normal in Fabric notebook environments and don't affect functionality — you can safely ignore them.

In [ ]:
%pip install semantic-link --quiet --disable-pip-version-check

import requests, json, base64, time, random, uuid
from datetime import datetime, timedelta
import sempy.fabric as fabric
from notebookutils import mssparkutils
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp

workspace_id = fabric.get_workspace_id()
token = mssparkutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
print(f"Workspace ID: {workspace_id}\n")

# --- Lakehouse -----------------------------------------------------------------
print("1. Checking for LamnaHealthcareLH...")
resp = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/lakehouses",
    headers=headers
)
lakehouses = resp.json().get("value", [])
lh = next((l for l in lakehouses if l["displayName"] == "LamnaHealthcareLH"), None)

if lh:
    lakehouse_id = lh["id"]
    print(f"   Found existing LamnaHealthcareLH (id: {lakehouse_id})")
else:
    print("   Creating LamnaHealthcareLH...")
    lakehouse_id = fabric.create_lakehouse(
        display_name="LamnaHealthcareLH",
        description="Shared healthcare lakehouse for Fabric IQ hands-on lab",
        enable_schema=False
    )
    print(f"   Created LamnaHealthcareLH (id: {lakehouse_id})")

lakehouse_tables_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables"

# --- Eventhouse ----------------------------------------------------------------
print("\n2. Checking for LamnaHealthcareEH...")
resp = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/eventhouses",
    headers=headers
)
eventhouses = resp.json().get("value", [])
eh = next((e for e in eventhouses if e["displayName"] == "LamnaHealthcareEH"), None)

if eh:
    eventhouse_id = eh["id"]
    print(f"   Found existing LamnaHealthcareEH (id: {eventhouse_id})")
else:
    print("   Creating LamnaHealthcareEH...")
    resp = requests.post(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/eventhouses",
        headers=headers,
        json={"displayName": "LamnaHealthcareEH"}
    )
    eventhouse_id = resp.json()["id"]
    print(f"   Created LamnaHealthcareEH (id: {eventhouse_id})")
    print("   Waiting for eventhouse to initialize...")
    time.sleep(10)

# Get eventhouse query URI
resp = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/eventhouses/{eventhouse_id}",
    headers=headers
)
query_uri = resp.json()["properties"]["queryServiceUri"]
print(f"\n   Eventhouse query URI: {query_uri}")

spark = SparkSession.builder.getOrCreate()

print(f"\n{'='*60}")
print(f"Infrastructure ready!")
print(f"   Lakehouse : LamnaHealthcareLH  ({lakehouse_id})")
print(f"   Eventhouse: LamnaHealthcareEH  ({eventhouse_id})")
print(f"   OneLake   : {lakehouse_tables_path}")
print(f"{'='*60}")

## Step 1: Write lakehouse tables

Writes all 5 healthcare tables to the shared lakehouse (overwrite mode — safe to re-run).

In [ ]:
print("Writing lakehouse tables...\n")

# --- Hospitals -----------------------------------------------------------------
hospitals_data = [(1, "Lamna Healthcare Regional Medical Center", "Seattle", "WA")]
hospitals_schema = StructType([
    StructField("HospitalId",   IntegerType(), False),
    StructField("HospitalName", StringType(),  False),
    StructField("City",         StringType(),  False),
    StructField("State",        StringType(),  False)
])
spark.createDataFrame(hospitals_data, hospitals_schema) \
    .write.mode("overwrite").format("delta").save(f"{lakehouse_tables_path}/Hospitals")
print("OK Hospitals          (1 row)")

# --- Departments ---------------------------------------------------------------
departments_data = [
    (1, "Intensive Care Unit",    1, 3),
    (2, "Emergency Department",   1, 1),
    (3, "Surgical Services",      1, 2)
]
departments_schema = StructType([
    StructField("DepartmentId",   IntegerType(), False),
    StructField("DepartmentName", StringType(),  False),
    StructField("HospitalId",     IntegerType(), False),
    StructField("Floor",          IntegerType(), False)
])
spark.createDataFrame(departments_data, departments_schema) \
    .write.mode("overwrite").format("delta").save(f"{lakehouse_tables_path}/Departments")
print("OK Departments        (3 rows)")

# --- Rooms ---------------------------------------------------------------------
rooms_data = [
    (1,  "ICU-301", 1, "Critical Care"),
    (2,  "ICU-302", 1, "Critical Care"),
    (3,  "ICU-303", 1, "Critical Care"),
    (4,  "ER-101",  2, "Emergency"),
    (5,  "ER-102",  2, "Emergency"),
    (6,  "ER-103",  2, "Emergency"),
    (7,  "SUR-201", 3, "Post-Op"),
    (8,  "SUR-202", 3, "Post-Op"),
    (9,  "SUR-203", 3, "Post-Op"),
    (10, "SUR-204", 3, "Post-Op")
]
rooms_schema = StructType([
    StructField("RoomId",       IntegerType(), False),
    StructField("RoomNumber",   StringType(),  False),
    StructField("DepartmentId", IntegerType(), False),
    StructField("RoomType",     StringType(),  False)
])
spark.createDataFrame(rooms_data, rooms_schema) \
    .write.mode("overwrite").format("delta").save(f"{lakehouse_tables_path}/Rooms")
print("OK Rooms              (10 rows)")

# --- Patients ------------------------------------------------------------------
# Generate admission dates dynamically (today minus 5..1 days)
admission_dates = [(datetime.now() - timedelta(days=i)).strftime("%Y-%m-%d") for i in range(5, 0, -1)]
patients_data = [
    (1001, "Kerry",  "Allen",    "1965-03-15", admission_dates[0], 1),
    (1002, "Casey",  "Morgan",   "1978-07-22", admission_dates[1], 2),
    (1003, "Elijah", "Thompson", "1952-11-08", admission_dates[2], 3),
    (1004, "Priya",  "Desai",    "1988-05-19", admission_dates[3], 7),
    (1005, "Maya",   "Robinson", "1971-09-30", admission_dates[4], 8)
]
df_patients = spark.createDataFrame(patients_data, StructType([
    StructField("PatientId",     IntegerType(), False),
    StructField("FirstName",     StringType(),  False),
    StructField("LastName",      StringType(),  False),
    StructField("DateOfBirth",   StringType(),  False),
    StructField("AdmissionDate", StringType(),  False),
    StructField("CurrentRoomId", IntegerType(), False)
]))
df_patients = df_patients \
    .withColumn("DateOfBirth", to_timestamp("DateOfBirth", "yyyy-MM-dd")) \
    .withColumn("AdmissionDate", to_timestamp("AdmissionDate", "yyyy-MM-dd"))
df_patients.write.mode("overwrite").format("delta").save(f"{lakehouse_tables_path}/Patients")
print("OK Patients           (5 rows)")

# --- VitalSignEquipment --------------------------------------------------------
equipment_data = [
    ("VS-1001", 1001, "Continuous Monitoring", admission_dates[0]),
    ("VS-1002", 1002, "Continuous Monitoring", admission_dates[1]),
    ("VS-1003", 1003, "Continuous Monitoring", admission_dates[2]),
    ("VS-1004", 1004, "Continuous Monitoring", admission_dates[3]),
    ("VS-1005", 1005, "Continuous Monitoring", admission_dates[4])
]
df_equipment = spark.createDataFrame(equipment_data, StructType([
    StructField("EquipmentId",          StringType(),  False),
    StructField("PatientId",            IntegerType(), False),
    StructField("EquipmentType",        StringType(),  False),
    StructField("MonitoringStartDate",  StringType(),  False)
]))
df_equipment = df_equipment.withColumn("MonitoringStartDate", to_timestamp("MonitoringStartDate", "yyyy-MM-dd"))
df_equipment.write.mode("overwrite").format("delta").save(f"{lakehouse_tables_path}/VitalSignEquipment")
print("OK VitalSignEquipment (5 rows)")

print("\nAll lakehouse tables written!")

## Step 2: Ingest VitalSignsReadings to eventhouse

Creates the eventhouse table and ingests time-series vital signs data (15 rows). Skips ingestion if data already exists.

In [ ]:
db = "LamnaHealthcareEH"

kql_token = mssparkutils.credentials.getToken(query_uri)
kql_headers = {"Authorization": f"Bearer {kql_token}", "Content-Type": "application/json"}

def kql_mgmt(command):
    resp = requests.post(f"{query_uri}/v1/rest/mgmt", headers=kql_headers, json={"db": db, "csl": command})
    resp.raise_for_status()
    return resp.json()

def kql_query(query):
    resp = requests.post(f"{query_uri}/v1/rest/query", headers=kql_headers, json={"db": db, "csl": query})
    resp.raise_for_status()
    tables = resp.json().get("Tables", [])
    return tables[0]["Rows"] if tables else []

# Create table
kql_mgmt("""
.create-merge table VitalSignsReadings (
    ReadingId:        int,
    EquipmentId:      string,
    Timestamp:        datetime,
    HeartRate:        int,
    OxygenSaturation: int,
    RespiratoryRate:  int
)
""")
print("OK VitalSignsReadings table created / verified")

# Check if data exists
rows = kql_query("VitalSignsReadings | count")
existing_count = rows[0][0] if rows else 0

if existing_count > 0:
    print(f"VitalSignsReadings already has {existing_count} rows - skipping ingestion")
else:
    # Generate timestamps dynamically (today at noon UTC)
    today = datetime.now().strftime("%Y-%m-%d")

    # Generate readings with 5-minute increments (3 readings per equipment = 15 total)
    readings = []
    readings.append(f"1,VS-1001,{today}T12:00:00Z,70,95,13")
    readings.append(f"2,VS-1001,{today}T12:05:00Z,72,95,13")
    readings.append(f"3,VS-1001,{today}T12:10:00Z,71,96,14")
    readings.append(f"4,VS-1002,{today}T12:00:00Z,68,97,12")
    readings.append(f"5,VS-1002,{today}T12:05:00Z,67,97,12")
    readings.append(f"6,VS-1002,{today}T12:10:00Z,69,98,13")
    readings.append(f"7,VS-1003,{today}T12:00:00Z,65,95,13")
    readings.append(f"8,VS-1003,{today}T12:05:00Z,66,94,14")
    readings.append(f"9,VS-1003,{today}T12:10:00Z,67,96,14")
    readings.append(f"10,VS-1004,{today}T12:00:00Z,95,98,16")
    readings.append(f"11,VS-1004,{today}T12:05:00Z,98,98,16")
    readings.append(f"12,VS-1004,{today}T12:10:00Z,100,97,17")
    readings.append(f"13,VS-1005,{today}T12:00:00Z,78,96,15")
    readings.append(f"14,VS-1005,{today}T12:05:00Z,77,97,15")
    readings.append(f"15,VS-1005,{today}T12:10:00Z,79,97,14")

    readings_csv = "\n".join(readings)
    kql_mgmt(f".ingest inline into table VitalSignsReadings <|\n{readings_csv}")
    print("OK Ingested 15 rows into VitalSignsReadings")

print("\nEventhouse step complete!")

## Done — shared data is ready

The workspace now contains the shared items that participants will bind their own ontologies to:

| Item | Type | Notes |
| --- | --- | --- |
| `LamnaHealthcareLH` | Lakehouse | Tables: `Hospitals`, `Departments`, `Rooms`, `Patients`, `VitalSignEquipment` |
| `LamnaHealthcareEH` | Eventhouse | KQL table: `VitalSignsReadings` (time-series) |

**Next:** grant all 25 participants access to the workspace (see `setup/instructor-setup-guide.md`), then hand out `lab/fabric-iq-hands-on-lab.md`.

> Do **not** run the ontology-creation steps here. Each participant creates their own `LamnaHealthcareOntology_<ID>` and `LamnaHealthcareAgent_<ID>` during the lab.